In [ ]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [ ]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

In [ ]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
    return accuracies, losses

In [ ]:

epochs = 5
model = MiniCNN()
model.to(device)
train_set = "en attente"

training_curves, training_curves = train_crossEntropyLoss(train_set,model, epochs)


In [ ]:
torch.save(model.state_dict(), f"model/trained_model/MiniCNN_{epochs}_epochs.pth")